# Faruq-v3 — frozen ACMC synthetic-density diagnostic

Meregenerasi B0–B3 dari identity **validation Faruq-v3 saja**, lalu membandingkan D0FT dan ACMC1 seed 42 tanpa training dan tanpa membuka test. Hasil hanya development diagnostic dan tidak mengubah locked-test `NOT_CONFIRMED`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REQUIRED = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-acmc-optimization-control-v1/D0FT_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
D0FT = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
ACMC = require_project_artifact(PROJECT_ROOT, REQUIRED[2])
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
GROUPED_MANIFEST = DATA_ROOT / 'faruq_grouped_manifest.json'
BENCHMARK_ROOT = PROJECT_ROOT / 'benchmarks/faruq-v3-synthetic-density-v1'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-synthetic-density-v1'
print('GPU      :', torch.cuda.get_device_name(0))
print('PROJECT  :', PROJECT_ROOT)
print('BENCHMARK:', BENCHMARK_ROOT)

In [ ]:
setup_command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_synthetic_density_setup',
    '--data-root', str(DATA_ROOT), '--grouped-summary', str(GROUPED_SUMMARY),
    '--grouped-manifest', str(GROUPED_MANIFEST), '--output-root', str(BENCHMARK_ROOT),
    '--scenes-per-condition', '100', '--seed', '42',
]
print('MENYIAPKAN BENCHMARK:', ' '.join(setup_command), flush=True)
subprocess.run(setup_command, cwd=REPO, check=True)

In [ ]:
import pandas as pd
from IPython.display import display
SETUP = BENCHMARK_ROOT / 'setup_summary.json'
setup = json.loads(SETUP.read_text())
audit = json.loads(Path(setup['validation_library_audit']).read_text())
print('LIBRARY AUDIT:', audit['gates'])
assert audit['safe_for_development_diagnostic'] is True
display(pd.DataFrame([{'condition': name, 'density': row['density'], 'scenes': setup['scenes_per_condition']} for name, row in setup['arms'].items()]))
print('TRAINING:', setup['training_executed'], '| TEST:', setup['test_images_accessed'])

In [ ]:
screen_command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_synthetic_density_screening',
    '--setup-summary', str(SETUP), '--d0ft-checkpoint', str(D0FT),
    '--acmc-checkpoint', str(ACMC), '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0',
]
LOG = OUTPUT_ROOT / 'synthetic_density_run.log'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('MENJALANKAN SCREENING:', ' '.join(screen_command), flush=True)
process = subprocess.Popen(screen_command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with LOG.open('a', encoding='utf-8') as log:
    for line in process.stdout:
        print(line, end='', flush=True); log.write(line); log.flush()
return_code = process.wait()
if return_code != 0:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-100:]))
    raise RuntimeError(f'Screening gagal: {return_code}; log={LOG}')

In [ ]:
SUMMARY = OUTPUT_ROOT / 'synthetic_density_seed42_summary.json'
result = json.loads(SUMMARY.read_text())
table = pd.DataFrame(result['rows'])
percent = [column for column in table.columns if column.startswith(('d0ft_', 'acmc1_', 'delta_'))]
display(table.style.format({column: '{:+.2%}' if column.startswith('delta_') else '{:.2%}' for column in percent}))
print('SUMMARY:', result['summary'])
print('STATUS :', result['interpretation_status'])
print('TRAINING:', result['training_executed'], '| TEST:', result['test_images_accessed'])
print('Kirim tabel ini. Jangan mengubah locked-test conclusion atau melakukan tuning.')